In [ ]:
# Example 1: Basic Incremental XML Writing

from et_xmlfile import xmlfile
import tempfile
import os

print("=== Example 1: Basic Incremental XML Writing ===")

try:
    # Create a temporary file for XML output
    with tempfile.NamedTemporaryFile(mode='w+b', suffix='.xml', delete=False) as temp_file:
        temp_filename = temp_file.name
    
    print(f"✓ Temporary XML file: {temp_filename}")
    
    # Write XML incrementally
    with xmlfile(temp_filename) as xf:
        with xf.element('library'):
            # Add multiple books
            for i in range(3):
                with xf.element('book', id=str(i+1)):
                    with xf.element('title'):
                        xf.write(f'Book Title {i+1}')
                    with xf.element('author'):
                        xf.write(f'Author {i+1}')
                    with xf.element('year'):
                        xf.write(str(2020 + i))
    
    print("✓ XML written incrementally")
    
    # Read and display the generated XML
    with open(temp_filename, 'r') as f:
        xml_content = f.read()
    
    print(f"✓ Generated XML ({len(xml_content)} characters):")
    print(xml_content[:200] + "..." if len(xml_content) > 200 else xml_content)
    
    # Clean up
    os.unlink(temp_filename)
    print("✓ Cleanup completed")

except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"et_xmlfile example (cell 0) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed successfully\n")

In [ ]:
# Example 5: Complex Nested Structures and Error Handling

from et_xmlfile import xmlfile
from io import BytesIO

print("=== Example 5: Complex Nested Structures ===")

try:
    xml_buffer = BytesIO()
    
    with xmlfile(xml_buffer) as xf:
        with xf.element('organization'):
            # Departments with nested structure
            departments = {
                'Engineering': ['Alice', 'Bob', 'Charlie'],
                'Marketing': ['David', 'Eve'],
                'HR': ['Frank']
            }
            
            for dept_name, employees in departments.items():
                with xf.element('department', name=dept_name):
                    with xf.element('head_count'):
                        xf.write(str(len(employees)))
                    
                    with xf.element('employees'):
                        for emp in employees:
                            with xf.element('employee'):
                                with xf.element('name'):
                                    xf.write(emp)
                                with xf.element('id'):
                                    xf.write(f"EMP{hash(emp) % 10000:04d}")
                                
                                # Nested contact info
                                with xf.element('contact'):
                                    with xf.element('email'):
                                        xf.write(f"{emp.lower()}@company.com")
    
    xml_content = xml_buffer.getvalue().decode('utf-8')
    print(f"✓ Complex nested XML generated ({len(xml_content)} characters)")
    
    # Analyze structure
    dept_count = xml_content.count('<department')
    emp_count = xml_content.count('<employee>')
    nest_levels = xml_content.count('<contact>')
    
    print(f"✓ Structure: {dept_count} departments, {emp_count} employees")
    print(f"✓ Nested levels: {nest_levels} contact sections")
    # 3 departments holding 6 employees, each with one contact block
    assert (dept_count, emp_count, nest_levels) == (3, 6, 6), (dept_count, emp_count, nest_levels)
    print(f"Sample: {xml_content[:120]}...")
    
    print("\n--- Minimal Document Test ---")
    test_buffer = BytesIO()
    with xmlfile(test_buffer) as xf:
        with xf.element('test'):
            xf.write("Valid content")
    assert test_buffer.getvalue() == b"<test>Valid content</test>", test_buffer.getvalue()
    print("✓ Normal operation successful")

except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"et_xmlfile example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 completed successfully\n")

In [ ]:
# Example 4: Large Dataset Streaming

from et_xmlfile import xmlfile
from io import BytesIO
import time

print("=== Example 4: Large Dataset Streaming ===")

try:
    xml_buffer = BytesIO()
    start_time = time.time()
    
    with xmlfile(xml_buffer) as xf:
        with xf.element('dataset', size="large"):
            # Simulate processing large amount of data
            with xf.element('records'):
                for i in range(100):  # Simulate 100 records
                    with xf.element('record', id=f"R{i:03d}"):
                        with xf.element('timestamp'):
                            xf.write(f"2024-01-{(i%30)+1:02d}T{(i%24):02d}:00:00Z")
                        with xf.element('value'):
                            xf.write(str(i * 3.14159))
                        with xf.element('status'):
                            xf.write("active" if i % 2 == 0 else "inactive")
    
    processing_time = time.time() - start_time
    xml_content = xml_buffer.getvalue().decode('utf-8')
    
    print(f"✓ Large dataset XML generated in {processing_time:.3f}s")
    print(f"✓ Total size: {len(xml_content)} characters")
    print(f"✓ Records processed: {xml_content.count('<record')}")
    print(f"Sample content: {xml_content[:100]}...")
    
    # Memory efficiency check
    buffer_size = xml_buffer.tell()
    print(f"✓ Buffer size: {buffer_size} bytes")

except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"et_xmlfile example (cell 2) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 completed successfully\n")

In [ ]:
# Example 3: XML with Attributes and Namespaces

from et_xmlfile import xmlfile
from io import BytesIO

print("=== Example 3: XML with Attributes and Namespaces ===")

try:
    xml_buffer = BytesIO()
    
    with xmlfile(xml_buffer) as xf:
        # Create root element with namespace
        with xf.element('document', 
                       xmlns="http://example.com/doc",
                       version="1.0"):
            
            # Add metadata section
            with xf.element('metadata'):
                with xf.element('created', format="ISO"):
                    xf.write("2024-01-15T10:30:00Z")
                with xf.element('author', role="editor"):
                    xf.write("John Doe")
            
            # Add content with various attributes
            with xf.element('content'):
                sections = [
                    {'title': 'Introduction', 'level': '1', 'pages': '5'},
                    {'title': 'Analysis', 'level': '1', 'pages': '12'},
                    {'title': 'Conclusion', 'level': '1', 'pages': '3'}
                ]
                
                for section in sections:
                    with xf.element('section', 
                                   level=section['level'],
                                   pages=section['pages']):
                        with xf.element('title'):
                            xf.write(section['title'])
    
    xml_content = xml_buffer.getvalue().decode('utf-8')
    print(f"✓ XML with attributes generated ({len(xml_content)} characters)")
    print(f"Content preview: {xml_content[:150]}...")
    
    # Count attributes
    attr_count = xml_content.count('=')
    print(f"✓ Contains {attr_count} attributes")

except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"et_xmlfile example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 completed successfully\n")

In [ ]:
# Example 2: In-Memory XML Generation

from et_xmlfile import xmlfile
from io import BytesIO

print("=== Example 2: In-Memory XML Generation ===")

try:
    # Create XML in memory using BytesIO
    xml_buffer = BytesIO()
    
    with xmlfile(xml_buffer) as xf:
        with xf.element('products'):
            # Generate product catalog
            products = [
                {'id': 'P001', 'name': 'Laptop', 'price': 999.99},
                {'id': 'P002', 'name': 'Mouse', 'price': 25.50},
                {'id': 'P003', 'name': 'Keyboard', 'price': 75.00}
            ]
            
            for product in products:
                with xf.element('product', id=product['id']):
                    with xf.element('name'):
                        xf.write(product['name'])
                    with xf.element('price', currency='USD'):
                        xf.write(f"{product['price']:.2f}")
    
    # Get XML content from buffer
    xml_content = xml_buffer.getvalue().decode('utf-8')
    print(f"✓ In-memory XML generated ({len(xml_content)} characters)")
    print(f"Content: {xml_content}")
    
    # Verify XML structure
    print(f"✓ Contains {xml_content.count('<product')} products")

except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"et_xmlfile example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed successfully\n")